In [1]:
with open('./files/input.txt', 'r') as file:
    text = file.read()
    
print(f" length of text: {len(text)}")

 length of text: 1115394


In [2]:
print(text[:1000])  # Print first 1000 characters of the text

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocabulary size:", vocab_size)
print("Characters:", ''.join(chars))

Vocabulary size: 65
Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [4]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [ stoi[c] for c in s ] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([ itos[i] for i in l ]) # decoder

print(encode("hello world"))
print(decode(encode("hello world")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [5]:
import torch
data = torch.tensor(encode(text), dtype=torch.float32)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.float32
tensor([18., 47., 56., 57., 58.,  1., 15., 47., 58., 47., 64., 43., 52., 10.,
         0., 14., 43., 44., 53., 56., 43.,  1., 61., 43.,  1., 54., 56., 53.,
        41., 43., 43., 42.,  1., 39., 52., 63.,  1., 44., 59., 56., 58., 46.,
        43., 56.,  6.,  1., 46., 43., 39., 56.,  1., 51., 43.,  1., 57., 54.,
        43., 39., 49.,  8.,  0.,  0., 13., 50., 50., 10.,  0., 31., 54., 43.,
        39., 49.,  6.,  1., 57., 54., 43., 39., 49.,  8.,  0.,  0., 18., 47.,
        56., 57., 58.,  1., 15., 47., 58., 47., 64., 43., 52., 10.,  0., 37.,
        53., 59.,  1., 39., 56., 43.,  1., 39., 50., 50.,  1., 56., 43., 57.,
        53., 50., 60., 43., 42.,  1., 56., 39., 58., 46., 43., 56.,  1., 58.,
        53.,  1., 42., 47., 43.,  1., 58., 46., 39., 52.,  1., 58., 53.,  1.,
        44., 39., 51., 47., 57., 46., 12.,  0.,  0., 13., 50., 50., 10.,  0.,
        30., 43., 57., 53., 50., 60., 43., 42.,  8.,  1., 56., 43., 57., 53.,
        50., 60., 43., 42., 

In [8]:
n = int(0.9 * len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8  # context length for predictions
train_data[:block_size+1]

tensor([18., 47., 56., 57., 58.,  1., 15., 47., 58.])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context.tolist()} the target: {target.item()}")


when input is [18.0] the target: 47.0
when input is [18.0, 47.0] the target: 56.0
when input is [18.0, 47.0, 56.0] the target: 57.0
when input is [18.0, 47.0, 56.0, 57.0] the target: 58.0
when input is [18.0, 47.0, 56.0, 57.0, 58.0] the target: 1.0
when input is [18.0, 47.0, 56.0, 57.0, 58.0, 1.0] the target: 15.0
when input is [18.0, 47.0, 56.0, 57.0, 58.0, 1.0, 15.0] the target: 47.0
when input is [18.0, 47.0, 56.0, 57.0, 58.0, 1.0, 15.0, 47.0] the target: 58.0


In [13]:
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)   
print(xb)
print('targets:')
print(yb.shape)
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target.item()}")

inputs:
torch.Size([4, 8])
tensor([[ 1., 41., 39., 50., 50., 57.,  1., 39.],
        [37., 53., 59.,  1., 57., 46., 53., 59.],
        [53., 59.,  1., 39., 56., 58.,  1., 42.],
        [59., 50.,  0., 18., 53., 56., 58., 46.]])
targets:
torch.Size([4, 8])
tensor([[41., 39., 50., 50., 57.,  1., 39., 45.],
        [53., 59.,  1., 57., 46., 53., 59., 58.],
        [59.,  1., 39., 56., 58.,  1., 42., 43.],
        [50.,  0., 18., 53., 56., 58., 46., 61.]])
when input is [1.0] the target: 41.0
when input is [1.0, 41.0] the target: 39.0
when input is [1.0, 41.0, 39.0] the target: 50.0
when input is [1.0, 41.0, 39.0, 50.0] the target: 50.0
when input is [1.0, 41.0, 39.0, 50.0, 50.0] the target: 57.0
when input is [1.0, 41.0, 39.0, 50.0, 50.0, 57.0] the target: 1.0
when input is [1.0, 41.0, 39.0, 50.0, 50.0, 57.0, 1.0] the target: 39.0
when input is [1.0, 41.0, 39.0, 50.0, 50.0, 57.0, 1.0, 39.0] the target: 45.0
when input is [37.0] the target: 53.0
when input is [37.0, 53.0] the target: 59.0
